In [3]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

FREIHAND_RGB_DIR  = Path("/scratch/zt1/project/msml612/user/rdawkhar/AirSketch/data/raw/freihand/training/rgb")
LANDMARKS_PATH    = Path("/scratch/zt1/project/msml612/user/rdawkhar/AirSketch/data/processed/freihand/landmarks_2d.npy")
SPLITS_PATH       = Path("/scratch/zt1/project/msml612/user/rdawkhar/AirSketch/data/splits/freihand_splits.json")
IMAGE_SIZE = 224
INDEX_FINGERTIP = 8

# Landmark connectivity for drawing the hand skeleton
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),        # index
    (0, 9), (9, 10), (10, 11), (11, 12),   # middle
    (0, 13), (13, 14), (14, 15), (15, 16), # ring
    (0, 17), (17, 18), (18, 19), (19, 20), # pinky
    (5, 9), (9, 13), (13, 17),             # palm
]

In [ ]:
landmarks = np.load(LANDMARKS_PATH)
with SPLITS_PATH.open("r", encoding="utf-8") as f:
    splits = json.load(f)

print(f"Landmarks array shape: {landmarks.shape}")
print(f"Train: {len(splits['train']):,} images")
print(f"Val:   {len(splits['val']):,} images")
print(f"Test:  {len(splits['test']):,} images")

# Overlap check
train_set = set(splits["train"])
val_set = set(splits["val"])
test_set = set(splits["test"])
assert not (train_set & val_set), "Train/val overlap detected"
assert not (train_set & test_set), "Train/test overlap detected"
assert not (val_set & test_set), "Val/test overlap detected"
print("\n[OK] No split overlap detected.")

In [5]:
def draw_hand(ax, image: np.ndarray, uv: np.ndarray, title: str) -> None:
    """Plot an image with overlaid hand landmarks and skeleton."""
    ax.imshow(image)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

    # Scale normalized coordinates back to pixel space.
    uv_px = uv * IMAGE_SIZE

    # Draw skeleton connections.
    for a, b in HAND_CONNECTIONS:
        ax.plot(
            [uv_px[a, 0], uv_px[b, 0]],
            [uv_px[a, 1], uv_px[b, 1]],
            color="#4A90D9",
            linewidth=0.8,
            alpha=0.7,
        )

    # Draw all landmarks as small dots.
    ax.scatter(
        uv_px[:, 0],
        uv_px[:, 1],
        s=12,
        c="#FFFFFF",
        edgecolors="#222222",
        linewidths=0.4,
        zorder=5,
    )

    # Highlight index fingertip (landmark 8) in red.
    ax.scatter(
        uv_px[INDEX_FINGERTIP, 0],
        uv_px[INDEX_FINGERTIP, 1],
        s=40,
        c="#E84040",
        edgecolors="#FFFFFF",
        linewidths=0.8,
        zorder=6,
    )

In [ ]:
NUM_SAMPLES = 5
rng = np.random.default_rng(42)

fig, axes = plt.subplots(
    nrows=3,
    ncols=NUM_SAMPLES,
    figsize=(NUM_SAMPLES * 3, 3 * 3),
)
fig.suptitle(
    "FreiHAND samples - white dots: all landmarks, red dot: index fingertip (landmark 8)",
    fontsize=10,
    y=1.01,
)

for row_idx, split_name in enumerate(["train", "val", "test"]):
    split_indices = splits[split_name]
    chosen = rng.choice(split_indices, size=NUM_SAMPLES, replace=False)

    for col_idx, img_idx in enumerate(chosen):
        img_path = FREIHAND_RGB_DIR / f"{img_idx:08d}.jpg"
        image = np.array(Image.open(img_path).convert("RGB"))
        uv = landmarks[img_idx]

        ax = axes[row_idx, col_idx]
        draw_hand(ax, image, uv, title=f"{split_name} #{img_idx}")

        if col_idx == 0:
            ax.set_ylabel(split_name.upper(), fontsize=11, fontweight="bold", labelpad=8)

plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/freihand_sanity_check.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Figure saved to report/figures/freihand_sanity_check.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Index fingertip (landmark 8) coordinate distributions per split", fontsize=11)

for ax, split_name in zip(axes, ["train", "val", "test"]):
    idx = splits[split_name]
    tips = landmarks[idx, INDEX_FINGERTIP, :]

    ax.scatter(tips[::50, 0], tips[::50, 1], alpha=0.15, s=2, c="#4A90D9")
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_xlabel("x (normalized)")
    ax.set_ylabel("y (normalized)")
    ax.set_title(f"{split_name} (n={len(idx):,})")
    ax.set_aspect("equal")

    print(
        f"{split_name:5s} x: {tips[:, 0].mean():.3f} +- {tips[:, 0].std():.3f} "
        f"y: {tips[:, 1].mean():.3f} +- {tips[:, 1].std():.3f}"
    )

plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/fingertip_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Verify all 4 augmentations of each scene are in the same split.
NUM_SCENES = 32_560
violations = []

for scene_id in range(NUM_SCENES):
    scene_images = [scene_id + aug * NUM_SCENES for aug in range(4)]
    split_membership = set()

    for img_idx in scene_images:
        if img_idx in train_set:
            split_membership.add("train")
        elif img_idx in val_set:
            split_membership.add("val")
        elif img_idx in test_set:
            split_membership.add("test")

    if len(split_membership) > 1:
        violations.append((scene_id, split_membership))

if violations:
    print(f"[FAIL] {len(violations)} scenes have augmentations in different splits")
    for scene_id, membership in violations[:5]:
        print(f"  Scene {scene_id}: {sorted(membership)}")
else:
    print(f"[OK] All {NUM_SCENES:,} scenes have all 4 augmentations in the same split.")
    print("[OK] No data leakage detected.")

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open("data/processed/egohands/clip_index.json") as f:
    clip_index = json.load(f)

clip_names = list(clip_index.keys())
det_rates  = [clip_index[c]["summary"]["detection_rate"] for c in clip_names]
activities = [c.split("_")[0] for c in clip_names]

color_map  = {"CHESS": "#4A90D9", "CARDS": "#E84040",
              "JENGA": "#2DB37A", "PUZZLE": "#E8A830"}
colors     = [color_map.get(a, "#888888") for a in activities]

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(range(len(clip_names)), det_rates, color=colors, edgecolor="none")
ax.axhline(y=0.80, color="red",    linestyle="--", linewidth=1.2, label="80% threshold")
ax.axhline(y=0.70, color="orange", linestyle="--", linewidth=1.0, label="70% exclusion threshold")
ax.set_xticks(range(len(clip_names)))
ax.set_xticklabels([c.replace("_", "\n") for c in clip_names],
                   fontsize=5, rotation=0)
ax.set_ylabel("Detection rate")
ax.set_ylim(0, 1.05)
ax.set_title(f"MediaPipe detection rate per EgoHands clip\n"
             f"Mean: {np.mean(det_rates)*100:.1f}%  |  "
             f"Clips below 70%: {sum(r < 0.70 for r in det_rates)}")
ax.legend(fontsize=9)

from matplotlib.patches import Patch
legend_patches = [Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=legend_patches + [
    plt.Line2D([0],[0], color="red",    linestyle="--", label="80% threshold"),
    plt.Line2D([0],[0], color="orange", linestyle="--", label="70% exclusion threshold"),
], fontsize=8)

plt.tight_layout()
plt.savefig("report/figures/egohands_detection_rates.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Clips above 80%: {sum(r >= 0.80 for r in det_rates)}")
print(f"Clips 70-80%:    {sum(0.70 <= r < 0.80 for r in det_rates)}")
print(f"Clips below 70%: {sum(r < 0.70 for r in det_rates)}  (will be excluded)")

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17),
]

sample_clips = list(clip_index.keys())[::16][:3]   # clips 0, 16, 32

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("EgoHands sample frames with MediaPipe landmarks\n"
             "(red dot = index fingertip, blue = skeleton)", fontsize=10)

for row, clip_name in enumerate(sample_clips):
    lm_path   = clip_index[clip_name]["landmarks_path"]
    landmarks = np.load(lm_path)   # (100, 21, 2)

    img_dir = Path("data/raw/egohands/_LABELLED_SAMPLES") / clip_name
    frames  = sorted(img_dir.glob("frame_*.jpg"))

    for col, frame_idx in enumerate([0, 49, 99]):
        ax   = axes[row, col]
        img  = np.array(Image.open(frames[frame_idx]).convert("RGB"))
        uv   = landmarks[frame_idx]

        ax.imshow(img)

        if not np.any(np.isnan(uv)):
            h, w = img.shape[:2]
            uv_px = uv * np.array([w, h])

            for (a, b) in HAND_CONNECTIONS:
                ax.plot([uv_px[a,0], uv_px[b,0]],
                        [uv_px[a,1], uv_px[b,1]],
                        "b-", linewidth=0.8, alpha=0.6)
            ax.scatter(uv_px[:, 0], uv_px[:, 1], s=10,
                       c="white", edgecolors="steelblue", linewidths=0.4, zorder=5)
            ax.scatter(uv_px[8, 0], uv_px[8, 1],
                       s=40, c="red", zorder=6)
            status = "detected"
        else:
            status = "no detection"

        ax.set_title(f"frame {frame_idx+1} — {status}", fontsize=7)
        ax.axis("off")

        if col == 0:
            ax.set_ylabel(clip_name.replace("_", "\n"), fontsize=6)

plt.tight_layout()
plt.savefig("report/figures/egohands_samples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
with open("data/splits/merged_train_split.json") as f:
    merged = json.load(f)

print("Overlap check result:")
print(f"  overlap_check_passed: {merged['overlap_check_passed']}")
print(f"\nTraining sources:")
print(f"  FreiHAND train: {len(merged['sources']['freihand']['train_indices']):,} frames")
print(f"  EgoHands:       {merged['sources']['egohands']['total_frames']:,} frames "
      f"across {merged['sources']['egohands']['total_clips']} clips")
print(f"\nTest source (held-out, never seen during training):")
print(f"  {merged['test_source']['note']}")

assert merged["overlap_check_passed"], "Overlap check failed — do not proceed to training"
print("\n✓ All checks passed. Safe to proceed to issue #9.")